# Lesson 4.2: Named Entity Recognition

## 📖 4 Follow Along — Using Named Entity Recognition

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Working with location names represents a unique computational problem. Since we do not know in advance what the names might be, and since some proper nouns can be both cities and people, (i.e. Jefferson, Washington, Lincoln, etc.) we need the computer to have some concept of what a place is. This is where language models come in. 

Language models are a form of **Machine Learning (ML)** — a branch of Artificial Intelligence (AI) in which a system learns patterns from large amounts of data rather than following hand-written rules. They have been trained on a lot of text. Through statistical inference they establish the different types of **entities** or words in a text. These can be **parts of speech** like a verb, noun, adjective etc. With this basic understanding of grammar, they can infer more complex **entities** people, places, and organizations. 

One of the main libraries that Python uses to do this is `sPacy`. The sample below goes through the basic procedure for extracting an entity. 

**Don't worry about how the code works for now, just look at the result**

The code passes through the sentence:
>The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson.



In [51]:
import spacy

# Load the small English model — "sm" = small, fast, good enough for demos
nlp = spacy.load('en_core_web_sm')

# Our example sentence — deliberately contains a person named Jefferson AND
# a place named Jefferson so we can see whether spaCy tells them apart
text = """The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson."""

# nlp() runs the full pipeline: tokenize → tag parts of speech → find entities
doc = nlp(text)

# This coding helps display the result in a human-readable format. Understanding how it works is not important.
print(f"Text:\n{text.strip()}\n")
print("Named entities:")
for ent in doc.ents:
    print(f"  {ent.text!r:30} {ent.label_!r:10} {spacy.explain(ent.label_)}")

Text:
The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson.

Named entities:
  'The University of Virginia'   'ORG'      Companies, agencies, institutions, etc.
  'Charlottesville'              'GPE'      Countries, cities, states
  'Jefferson'                    'PERSON'   People, including fictional
  'the 1950s'                    'DATE'     Absolute or relative dates or periods
  'William Faulkner'             'PERSON'   People, including fictional
  'Jefferson'                    'GPE'      Countries, cities, states



> 📊 **Output:** Notice how accurately spaCy is able to distinguish between an organization in Virginia (UVA), a person Jefferson, and the place Jefferson (geopolitical entity).

spaCy is only as accurate as the data provided to it. If the text data is garbled or too short, it will likely have trouble. Undoubtedly, there are sentences in our `sentences` column that are not going to be read properly, but what we are relying on is the sheer volume of text. Even with some false positives and false negatives, we should be able to build a pretty good overview of the most mentioned places.

## 📖 5 Follow Along — Extract Entities in All Sentences

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Doing one extraction on one sentence in `spacy` is pretty straight forward. We simply run the function `nlp()` on whatever sentence we want to analyze and save the result to a new variable, usually called `doc`. When we run this on a column with thousands of sentences, you start to run into performance issues because you are doing the procedure one at a time, and you also don't really know what's going on because there's no feedback. The functions below modify the above procedure a bit and basically asks your computer to use multiple processors, and it also provides a little progress bar. Finally, instead of using the very small model that we used above, we are now going to use a slightly bigger model called `en_core_web_md`, this will hopefully help us find more locations!

We are now ready to apply this function to `sentences` and create a new column called `toponyms`. 

**Warning this process will take a couple of minutes**

If this does not work, I have saved the results as `JMU_toponyms.pickle` in the `../data/JMU/` folder. You can simply keep running the code below on that imported file.

In [54]:
import spacy
from tqdm import tqdm

# Load the medium English model — more accurate than "sm" for location detection
nlp = spacy.load('en_core_web_md')

# nlp.pipe() processes all sentences in batches — much faster than one at a time.
# batch_size=256 means 256 sentences are grouped together per pass.
# tqdm displays a progress bar so you can track how far along the process is.
sentences = df_reddit_sentences['sentences']
toponyms = []

for doc in tqdm(nlp.pipe(sentences, batch_size=256), total=len(sentences)):
    # For each sentence, collect any entities labelled GPE (Geopolitical Entity = places)
    gpes = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
    toponyms.append(gpes if gpes else None)

# Store the results as a new column in the DataFrame
df_reddit_sentences['toponyms'] = toponyms

print(f"✅ Done — processed {len(sentences):,} sentences")

✅ Loaded spaCy model: en_core_web_md


100%|██████████| 30005/30005 [01:13<00:00, 409.00it/s]


> 📊 **Output:** You should see around 30,000 sentences processed in about 1–3 minutes. Keep in mind that this is an incredibly complex task for a computer. For every single sentence it has to: read the text, parse the grammar, decide whether any words are locations, extract those locations, and store the result — then repeat the whole process for the next sentence. The fact that it completes ~30,000 of these analyses in a few minutes is a remarkable feat of modern NLP. A human researcher doing this manually would need weeks.

### Analyze the results

Run the code below to show the table and the results. The display has been separated from the processing because you do not want to process the data every time you want to view the results. 

In [120]:
(
    df_reddit_sentences[['sentences', 'toponyms']]
    .sample(5, random_state=109)
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)

,sentences,toponyms
9215,"i eventually DID find a roommate and we get along extraordinarily well, but if you’d rather have a room by yourself or with someone better aligned to your identity- that’s something especially available as of recent.",None
4600,"Former Chesapeake resident, TIL Potomac is now Chandler.","['Chesapeake', 'TIL Potomac']"
10721,"I worked at the mall and during the winter my Soph and Senior years I worked at Massanutten and Wintergreen, mostly so I could ski for free and make a little extra money.",['Massanutten']
9303,Lots of group projects.,None
11318,"I already had it make my phone freeze after trying to verify, and then it wouldn't accept one verification so I had to say yes multiple times",None


> 📊 **Output:** Each row shows a sentence and the list of toponyms spaCy extracted from it. Some sentences have multiple place names; others may have extractions that look incorrect — that is expected at this scale. This sample confirms the NLP pipeline ran successfully.

> 💡 **Reflection:** Look at the locations `spaCy` found in this sample. Which ones could you have found with a simple keyword list — city names you already knew to search for? Which ones might you never have thought to include? And are there any sentences that clearly mention a place that `spaCy` missed entirely? Keep those examples in mind: they are the evidence for why NER is more powerful than a list, but also why it is not perfect.

Because not every sentence includes a toponym sometimes it will say `None`. We want to eliminate these rows because they are not relevant. Still, we might want to peek inside and calculate what percentage of sentences actually have toponyms. The calculation below achieves exactly this. It creates a table of the number of sentences with toponyms, and then divides the number of rows in that table by the total number of rows in the data set. This gives the percentage of rows that contain locations. In this case, around 4%.

In [128]:
# Filter to sentences that have at least one toponym — None rows are dropped here
df_reddit_toponyms = df_reddit_sentences[df_reddit_sentences['toponyms'].notna()]
total = len(df_reddit_sentences)

print(f"Sentences with at least one toponym: {len(df_reddit_toponyms):,} of {total:,} ({len(df_reddit_toponyms) / total * 100:.1f}%)")

Sentences with at least one toponym: 1,217 of 30,005 (4.1%)


> 💡 **Reflection:** Only about 4% of sentences contain a place name. This is a pretty low number. What does that tell you about how people write on Reddit? Are most posts about events and opinions rather than places? If we are only using a small subset of sentences, how might that distort our results when we look at sentiment by locations?


## 📖 6 Follow Along — Counting Toponyms

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

The code below counts how often each place name appears across the entire dataset. It does this in two steps:

1. **Flatten** — each row currently holds a *list* of toponyms (a sentence with two places has two items in its list). `.explode()` unpacks those lists so each toponym gets its own row, the same operation we used in Section 2.
2. **Count** — `.value_counts()` counts how many times each toponym appears and sorts the result from most to least common.

In [ ]:
# Step 1: Flatten the lists — each toponym gets its own row (same .explode() we used earlier)
unnested = df_reddit_toponyms['toponyms'].explode()

# Step 2: Count and sort — .value_counts() counts occurrences, most common first
toponym_counts_df = (
    unnested
    .value_counts()
    .reset_index()
    .rename(columns={'toponyms': 'Toponym', 'count': 'Count'})
)

toponym_counts_df.head(10)

,Toponym,Count
4,Harrisonburg,240
7,Virginia,114
8,VT,47
40,VA,42
19,US,41
72,Florida,21
66,harrisonburg,20
23,America,19
65,Breeze,19
47,FCS,19


> 📊 **Output:** The table lists each unique place name and how many times it appears across all sentences, sorted from most to least common. Scan the list for any entries that do not look like real place names — those are false positives from the NER model.

> 💡 **Reflection:** Not surprisingly, Harrisonburg is the top toponym. But look further down the list — there are also entries like `FCS` and `Breeze` that are not places at all. What other results look like odd to you? What might need to be fixed down the road? 

### 7 Visualizing Toponyms

A bar chart is a natural fit here — each bar represents one place name, and its height shows how often it appears. `px.bar()` takes the DataFrame we just built and maps `Toponym` to the x-axis and `Count` to the y-axis.

In [132]:
import plotly.express as px

# Take the top 10 most common toponyms for plotting
toponym_counts_top10 = toponym_counts_df.head(10)

# Create the bar chart using Plotly
fig = px.bar(
    toponym_counts_top10,
    x='Toponym',
    y='Count',
    title='Top 10 Most Common Toponyms',
    text='Count'
)

# Display the plot
fig.show()

> 📊 **Output:** The bar chart shows the ten most frequently mentioned toponyms. Each bar's height represents how many sentences contained that place name.

> 💡 **Reflection:** Take a look at some of the names that are synonyms. If you were to stack these bars up on top of each other, what would the chart look like? What three places dominate discussion on the reddit thread?

### 7.2 Engagement by Toponym

Raw counts tell us what places are *mentioned* most often. But do those same places also generate the most discussion — the posts that get the most upvotes?

To answer that, we need two numbers per place at the same time: how often it appears, and the average score of posts that mention it. The code below computes both in a single step using `.groupby().agg()`:

| Step | What it does |
|---|---|
| **`.explode()`** | Same as before — one toponym per row |
| **`.groupby('toponyms')`** | Group all rows that share the same place name |
| **`.agg(...)`** | For each group, compute multiple summary statistics at once — here, `count` and `mean` |

In [142]:
# Flatten: one toponym per row, keep the post score alongside it
toponym_score_df = (
    df_reddit_toponyms[['toponyms', 'score']]
    .explode('toponyms')
    .dropna(subset=['toponyms'])
)

# Group by toponym: count mentions AND average the post score
# Filter to places mentioned at least 10 times — rare mentions can have
# artificially high average scores, so a minimum count gives cleaner results
toponym_engagement = (
    toponym_score_df
    .groupby('toponyms', as_index=False)
    .agg(Count=('toponyms', 'count'), Avg_Score=('score', 'mean'))
    .rename(columns={'toponyms': 'Toponym'})
    .query('Count >= 10')
    .sort_values(['Avg_Score', 'Count'], ascending=False)
    .reset_index(drop=True)
)

toponym_engagement.head(15)

,Toponym,Count,Avg_Score
0,US,41,25.390244
1,Virginia,114,23.508772
2,Breeze,19,22.789474
3,Harrisonburg,240,20.316667
4,harrisonburg,20,17.700000
5,Florida,21,17.095238
6,VT,47,17.021277
7,Richmond,15,15.066667
8,Jersey,17,14.000000
9,VA,42,13.404762


> 📊 **Output:** The table shows toponyms ranked by engagement. `Count` is the number of times a place was mentioned; `Avg_Score` is the mean upvote score of posts that mentioned it. Places at the top were both frequently mentioned and generated high engagement.

In [146]:
# Take the top 30 by engagement for plotting
top30_engagement = toponym_engagement.head(30)

# Treemap: each box is one place name
#   Box SIZE  → Count        (bigger box = mentioned more often)
#   Box COLOR → Avg_Score    (darker blue = higher average upvote score)
fig = px.treemap(
    top30_engagement,
    path=['Toponym'],
    values='Count',
    color='Avg_Score',
    title='Toponyms: Box Size = Mentions · Color = Average Upvote Score',
    labels={'Avg_Score': 'Avg Score', 'Count': 'Mentions'},
    color_continuous_scale='Blues',
)

fig.update_traces(textinfo='label+value')
fig.show()

> 📊 **Output:** The treemap encodes two variables at once: box size represents how often a place was mentioned, and color depth shows the average upvote score of posts that mentioned it. Larger, darker boxes are both frequently mentioned and highly engaging.

> 💡 **Reflection:** In this chart, box size shows how often a place was mentioned and color shows how much engagement those posts generated. Which places are both large *and* dark — frequently mentioned and highly engaging? Are there any places with a small box but a deep color, meaning they come up rarely but generate a strong reaction when they do? What might explain that pattern?

## 📖 8 Follow Along — Save & Export

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

In [ ]:
# Save the filtered DataFrame (sentences that have toponyms) as a pickle.
# This preserves dtypes so the next notebook doesn't have to re-cast everything.
df_reddit_toponyms.to_pickle('../data/JMU/JMU_toponyms.pickle')
print("✅ Saved ../data/JMU/JMU_toponyms.pickle")

### 8.1 Prepare for Google Sheets Export

Before we export, we need to add two things:

1. A **`unique_id`** — a stable number for each row. This is critical because after your team edits the file in Google Sheets and re-downloads it, we need to be able to reconnect each reviewed row back to the original sentence and its score. Without this anchor the data can't be rejoined.

2. A **`school_name`** column — in this lesson we only have JMU data, but the full analysis compares multiple Virginia universities. Adding this column now means the exported format is consistent.

We also need to **explode** the `toponyms` column. Right now each row holds a *list* of places found in one sentence. Google Sheets can't work with lists in a cell, so we split it so that each place gets its own row.

In [ ]:
# Assign stable sequential IDs before any further filtering
df_reddit_toponyms = df_reddit_toponyms.reset_index(drop=True)
df_reddit_toponyms.insert(0, 'unique_id', df_reddit_toponyms.index)

# Add school_name — in the multi-school version this column already exists in the source CSV
if 'school_name' not in df_reddit_toponyms.columns:
    df_reddit_toponyms['school_name'] = 'JMU'

# Explode: one row per (sentence, extracted_place) pair
df_locations_raw = (
    df_reddit_toponyms[['unique_id', 'school_name', 'date', 'score', 'sentences', 'toponyms']]
    .explode('toponyms')
    .rename(columns={'sentences': 'sentence', 'toponyms': 'extracted_place'})
    .dropna(subset=['extracted_place'])
    .reset_index(drop=True)
)

df_locations_raw.head(10)

In [ ]:
# Export to CSV for Google Sheets
df_locations_raw.to_csv('../data/JMU/JMU_locations_raw.csv', index=False)

print(f"✅ Saved ../data/JMU/JMU_locations_raw.csv")
print(f"   Rows (one per sentence-place pair): {len(df_locations_raw):,}")
print(f"   Unique extracted place names:        {df_locations_raw['extracted_place'].nunique():,}")
print(f"   Columns: {list(df_locations_raw.columns)}")

---

## Lesson Summary

**Section 4 — Using Named Entity Recognition**
- `spacy.load('model_name')` — loads a pre-trained NLP model into a variable (`nlp`) ready to process text
- `spacy.explain('LABEL')` — returns a human-readable description of any spaCy entity or part-of-speech label

**Section 5 — Extract Entities in All `sentences`**
- `nlp.pipe(texts)` — runs the NLP model efficiently over a large list of strings in batches
- `ent.label_` — the entity type assigned by the model (e.g. `GPE` for geopolitical entities, `LOC` for locations)
- `ent.text` — the raw string the model identified as a named entity

**Section 6 — Counting Toponyms**
- `df['col'].value_counts()` — counts how often each unique value appears; useful for ranking place-name frequency

**Section 7 — Visualizing Toponyms**
- `px.bar(df, x=..., y=...)` — creates a bar chart from a DataFrame column; used here to rank toponyms by frequency and engagement

**Section 8 — Save & Export**
- `df.reset_index(drop=True)` — resets the row index after filtering or merging
- `df.to_csv('file.csv', index=False)` — saves the DataFrame to a CSV file ready for Google Sheets review

➡️ **Next:** [Lesson 4.3 — Geoparsing in Python](lesson_4_3_geoparsing_mapping.ipynb)